# Influence and small-study diagnostics

Everything here is computed from `(y_i, se_i)` pairs and the pooled estimate from
`meta.classical`; nothing re-implements the pooling. `PoolMethod` selects the pool each
diagnostic re-runs: `"fe"` (fixed effect) or a random-effects `tau` method (`"dl"`, `"pm"`,
`"reml"`).

| function | what it returns |
|---|---|
| `leave_one_out` | the pool without each study; influence `(μ̂ − μ̂₍₋ᵢ₎) / se(μ̂)` |
| `egger` | the regression `y_i/se_i = b₀ + b₁/se_i + ε`; `t = b₀/se(b₀)` on `k − 2` df |
| `funnel_data` | points and pseudo-confidence contours `μ̂ ± z_m · se` |
| `forest_data` | per-study Wald intervals, the pooled interval, the prediction interval |
| `baujat` | each study's contribution to `Q` against its influence on `μ̂` |

In [ ]:
import numpy as np

from axiom.core import Spec
from axiom.meta import (
    BaujatData, Corpus, EggerTest, ForestData, ForestRow, FunnelContour, FunnelData, LeaveOneOut,
    PoolMethod, StudyRecord, baujat, egger, fixed_effect, forest_data, funnel_data, leave_one_out,
    random_effects,
)

## Two corpora: symmetric, and with small-study asymmetry

Both have fifteen studies around a true effect of `0.5`. In the asymmetric one the imprecise
studies are shifted upward in proportion to their se — the pattern selective reporting produces.

In [ ]:
rng = np.random.default_rng(11)
k = 15
se = np.sort(rng.uniform(0.05, 0.4, k))
y_sym = 0.5 + se * rng.normal(size=k)
y_asym = 0.5 + 1.5 * se + se * rng.normal(size=k)
corpus = Corpus(records=tuple(
    StudyRecord(study=f"s{i}", contributor=f"c{i}", quantity="elasticity", estimate=float(y_sym[i]),
                se=float(se[i]), read="experiment", family="fertilizer")
    for i in range(k)
))
print("symmetric pooled:", round(random_effects(y_sym, se).estimate, 3), "| asymmetric pooled:", round(random_effects(y_asym, se).estimate, 3))

## `leave_one_out`

Influence is the shift of the pooled estimate, in units of its own se, when study `i` is
dropped. `tau2s` records each reduced pool's between-study variance (all zero under `"fe"`).

In [ ]:
method: PoolMethod = "reml"
loo: LeaveOneOut = leave_one_out(y_sym, se, method=method)
print(f"full: {loo.full_estimate:.4f} ± {loo.full_se:.4f}  {loo.full_interval}")
order = np.argsort(-np.abs(loo.influence))[:3]
for i in order:
    print(f"  drop s{i}: {loo.estimates[i]:.4f} ± {loo.ses[i]:.4f}  influence {loo.influence[i]:+.3f}  tau² {loo.tau2s[i]:.4f}")
print("fixed-effect LOO tau² all zero:", set(leave_one_out(y_sym, se, method="fe").tau2s) == {0.0})

## `egger`

Egger's intercept is the asymmetry statistic. On the symmetric corpus it is indistinguishable
from zero; on the asymmetric one it is not. The `interval` is `b₀ ± t_{k−2} · se(b₀)`
(labelled `wald`).

In [ ]:
for label, yy in (("symmetric", y_sym), ("asymmetric", y_asym)):
    eg: EggerTest = egger(yy, se)
    print(f"{label:11s} intercept {eg.intercept:+.3f} ± {eg.se:.3f}  t = {eg.t:+.2f} on {eg.df} df  p = {eg.p:.4f}  {eg.interval}")
    print(f"            slope (bias-adjusted effect) {eg.slope:.3f} ± {eg.slope_se:.3f}")

## `funnel_data`

Study points (`y` against `se`, with `precision = 1/se`) and closed-form contours
`pooled ± z_m · se` for `se` from 0 to `max(se)` at masses 0.9 / 0.95 / 0.99. Counting points
outside the 0.95 contour is a quick read of asymmetry.

In [ ]:
def outside_95(fd: FunnelData) -> int:
    c95: FunnelContour = next(c for c in fd.contours if c.mass == 0.95)
    z = (c95.upper[-1] - fd.pooled) / c95.se[-1]
    return int(np.sum(np.abs(np.asarray(fd.y) - fd.pooled) > z * np.asarray(fd.se)))

for label, yy in (("symmetric", y_sym), ("asymmetric", y_asym)):
    fd = funnel_data(yy, se, fixed_effect(yy, se), n_grid=25)
    print(f"{label:11s} pooled {fd.pooled:.3f} | contours at {[c.mass for c in fd.contours]} | outside 0.95: {outside_95(fd)} of {fd.k}")
print("bare float pooled also accepted:", funnel_data(y_sym, se, 0.5).pooled)

## `forest_data`

From a `Corpus` (labels are the study ids) or `(y, se)` arrays, plus the pooled fit. With a
`PooledEstimate` the normalized weights are reported per `ForestRow` and the prediction
interval comes from `classical.prediction_interval`.

In [ ]:
re = random_effects(y_sym, se, tau_method="reml")
forest: ForestData = forest_data(corpus, None, re)
row: ForestRow = forest.rows[0]
print(f"{row.label}: {row.estimate:.3f}  {row.interval}  weight {row.weight:.3f}")
print(f"pooled {forest.pooled_estimate:.3f}  {forest.pooled_interval}")
print("prediction:", forest.prediction_interval, "| tau²:", round(forest.tau2, 5))
print("weights sum:", round(sum(r.weight for r in forest.rows), 12), "| round-trips:", Spec.from_json(forest.to_json()) == forest)
print("from arrays with labels:", forest_data(y_sym[:3], se[:3], 0.5, labels=["a", "b", "c"]).rows[1].label)

## `baujat`

Under fixed-effect pooling: `x_i = w_i (y_i − μ̂)²` (contribution to `Q`) against
`y_i = (μ̂ − μ̂₍₋ᵢ₎)² / var(μ̂₍₋ᵢ₎)` (influence). The study in the top-right corner is the one
to look at.

In [ ]:
bj: BaujatData = baujat(y_asym, se)
top = int(np.argmax(np.asarray(bj.q_contribution) * np.asarray(bj.influence)))
print("k =", bj.k, "| Q contributions sum to Q:", round(sum(bj.q_contribution), 4), "vs", round(fixed_effect(y_asym, se).heterogeneity.q, 4))
print(f"most influential: s{top}  Q-contribution {bj.q_contribution[top]:.3f}  influence {bj.influence[top]:.3f}")